# OR Tools

Google OR Tools, a strong classical optimiser, is used to solve the problem instances.

Import modules, set constants.

In [ ]:
import time

from ortools.constraint_solver import pywrapcp, routing_enums_pb2

from classes.MyDataLogger import MyDataLogger, MySubDataLogger
from modules.config_graphs import LOCATIONS
from modules.helper_functions_tsp import (
    find_distances_array,
)

MULTIPLIER = 10 # need to multiply by 10 to avoid roundings
#ORTools expects integers, but numbers are given to one decimal place sometimes.
TIME = 60
TARGET = 'ORTools'

Instantiate the datalogger

In [2]:
datalogger = MyDataLogger()

Iterate over locations and calculate distance.

In [ ]:
sdl_list = []
for i, locations in enumerate(LOCATIONS):
    t0 = time.time()
    # instantiate sub data logger
    sdl_list.append(MySubDataLogger(runid=datalogger.runid))
    dist_array, best_dist = find_distances_array(locations, print_comments=True)

    dist = [ #convert distance from array to list, and scale to integers.
    [int(MULTIPLIER * d) for d in row]
    for row in dist_array
    ]

    manager = pywrapcp.RoutingIndexManager(len(dist), 1, 0)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_idx, to_idx):
        return dist[  # noqa: B023
            manager.IndexToNode(from_idx)  # noqa: B023
        ][
            manager.IndexToNode(to_idx)  # noqa: B023
        ]

    transit_callback = routing.RegisterTransitCallback(distance_callback)

    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    ) # find first solution - greedy heuristic
    search_parameters.local_search_metaheuristic = (
    routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    ) # find more accurate solution using guided search
    search_parameters.time_limit.seconds = TIME

    solution = routing.SolveWithParameters(search_parameters)
    #rescale
    best_dist_found = solution.ObjectiveValue() / MULTIPLIER

    print('Objective:', best_dist_found)

    t1 = time.time()
    sdl_list[i].locations = locations
    sdl_list[i].elapsed = t1 - t0
    sdl_list[i].best_dist_found = best_dist_found
    sdl_list[i].best_dist = best_dist
    sdl_list[i].target = TARGET
    sdl_list[i].save_results_to_csv()


SubDataLogger instantiated.  Run ID = 20260829-21-48-35 - 21-48-35
Reading distance data
Data will be read from filename networks\sim_dist_10_locs.txt.
It is known that the shortest distance is 290.2
Objective: 290.2
Saving data to results\results.csv
SubDataLogger instantiated.  Run ID = 20260829-21-48-35 - 21-49-35
Reading distance data
Data will be read from filename networks\dg11_d.txt.
It is known that the shortest distance is 253
Objective: 253.0
Saving data to results\results.csv
SubDataLogger instantiated.  Run ID = 20260829-21-48-35 - 21-50-35
Reading distance data
Data will be read from filename networks\sim_dist_12_locs.txt.
It is known that the shortest distance is 297.2
Objective: 297.2
Saving data to results\results.csv
SubDataLogger instantiated.  Run ID = 20260829-21-48-35 - 21-51-35
Reading distance data
Data will be read from filename networks\p01_d.txt.
It is known that the shortest distance is 291
Objective: 291.0
Saving data to results\results.csv
SubDataLogger ins